## Messages

Messages are the fundamental units of context for models in LangChain. They represents the input and the output of models, carrying both the content and the metadata needed to represent the state of the conversation when interacting with an LLM. Messages are objects that contain:

. Role - Identifies the message type (e.g. system, user).\
. Content - Represent the actual content of the message (like text, images, audio, documents, etc.)\
. Metadata - Optional field such as response information, message IDs, and token usage.

LangChain provides a standard message type that works across all models providers, ensuring consistent behaviour regardless of the model being called.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")

## Text Prompts

Text prompts are strings - ideal for straightforward generation tasks where you den't need to retain conversation history.

In [2]:
model.invoke("What is LangChain?")

AIMessage(content='<think>\nOkay, the user is asking, "What is LangChain?" I need to explain LangChain in a clear way. First, LangChain is a framework for building applications with large language models like GPT. Let me recall the key points.\n\nLangChain helps developers integrate LLMs into their apps. It provides tools for handling prompts, memory, data, and more. The main components might be things like prompts, chains, agents, memory, and callbacks. Each of these serves a specific function.\n\nWait, the user might be a developer looking to use LangChain, so they need to know what it can do. They might also want to know why it\'s useful compared to building everything from scratch. Maybe I should mention how it simplifies the process by offering pre-built modules.\n\nI should break down the components. Prompts are for structuring inputs to the model. Chains allow combining multiple steps. Agents help the model make decisions. Memory keeps track of past interactions. Data connectors

Use Text prompts when:

. You have a single, standalone request.\
. You don't need conversational history.\
. You want minimal code complexity.

## Message Prompts

Alternatively, you can pass in a list of messages to the model by providing a list of message objects.

Message Types:

. System message - Tells the model how to behave and provide context for interaction.\
. Human message - Represents user input and interaction with the model.\
. AI message - Responses generated by a model, including text content, tool calls, and metadata.\
. Tool message - Represents the outputs to tool calls.

### System Message

A System Message is a high-level instruction given to the AI model that defines its behavior, personality, constraints, and objectives. It acts as the foundation for how the model responds during a conversation.

### Human Message

A Human Message is the message sent by the user to the AI. It represents the user's query, command, or context that the model uses to generate a response.

### AI Message 

An AI Message is the output generated by the language model in response to a user's request. It is produced by considering the system instructions and the conversation context.

### Tool Message

A Tool Message is a message containing the output from an external tool or function used by the AI. It allows the model to access real-time data, perform calculations, query databases, or interact with external systems.

In [3]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are a poetry expert."),
    HumanMessage("Write a poem about the sea.")
]

response = model.invoke(messages)
response.content

'<think>\nOkay, the user wants a poem about the sea. Let me start by brainstorming some sea-related imagery. Waves, tides, marine life, maybe the different times of day over the sea. I should consider the mood—should it be calming, mysterious, powerful? Maybe a mix of those.\n\nI need to decide on the structure. Maybe a traditional rhyme scheme, like ABAB or AABB. Let me think about flow and rhythm. The sea can be unpredictable, so maybe varying line lengths could reflect that. Or keep it consistent for a lyrical feel.\n\nFirst stanza could introduce the vastness and timelessness of the sea. Words like ancient, endless, whisper. Maybe personify the sea as a living entity. Next stanza about the waves, their motion, maybe using metaphors like dancers or horses. \n\nInclude elements like the moon\'s reflection, tides pulling, ships sailing. Maybe touch on different aspects: surface vs. depth, known vs. unknown. Mention creatures of the deep to add mystery. \n\nThink about the sea\'s duali

In [ ]:
## Detailed info to the LLM through System Message
from langchain.messages import SystemMessage, HumanMessage

system_msg = SystemMessage("""
You are a senior python developer with expertise in web frameworks. Always provide code examples and explain the reasoning. Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("What are the advantages of using FastAPI over Flask?")
]
response = model.invoke(messages)
print(response.content)

In [4]:
## Message Metadata
human_msg = HumanMessage(
    content = "Hello!",
    name = "Alice",    # Optional: Identify different users.
    id = "msg-123",    # Optional: Unique identifier for the message.
)

In [5]:
response = model.invoke([
    human_msg
])
response.content

'<think>\nOkay, the user just said "Hello!" so I need to respond appropriately. Since this is the first message, I should greet them back in a friendly manner. Let me think of a welcoming response. Maybe something like, "Hello! Welcome to the chat. How can I assist you today?" That sounds good. It\'s polite and opens the door for them to ask for help. I should check if there\'s any other info needed, but since there\'s no context, keeping it simple is best. Alright, that should work.\n</think>\n\nHello! Welcome to the chat. How can I assist you today? 😊'

In [9]:
from langchain.messages import SystemMessage, HumanMessage

# Create an AI message manually (e.g. for conversation history)
ai_msg = AIMessage(
    content="I'm a helpful assistant.",
    name="Assistant",
    id="msg-456"
)

# Add a conversational history
messages = [
    SystemMessage("You are a helpful assistant."),
    HumanMessage("Can you help me?"),
    ai_msg,   # Insert as if it came from the model
    HumanMessage("Great! What's 2 + 2?")
]
response = model.invoke(messages)
print(response.content)

<think>
Okay, the user just asked "What's 2 + 2?" Let me think about how to respond.

First, this is a straightforward arithmetic question. The answer is obviously 4. But maybe I should consider why they asked. Sometimes people ask this as a test or to check if someone is paying attention. Since the user previously said "Great! Can you help me?" after I responded to their initial "Can you help me?" maybe they're just testing the interaction.

I need to make sure my answer is clear and helpful. Let me confirm the calculation again. 2 plus 2 is indeed 4. There's no trick here unless the user is trying to see if I'll overcomplicate it. But since they asked directly, a simple answer should suffice.

I should also keep the tone friendly and offer further help in case they have more questions. Let me phrase it in a way that's concise and correct.
</think>

2 + 2 equals **4**! Let me know if you need help with anything else. 😊


In [11]:
response.usage_metadata  # Access metadata from the response

{'input_tokens': 50, 'output_tokens': 216, 'total_tokens': 266}

In [12]:
from langchain.messages import AIMessage
from langchain.messages import ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the message for illustration)
ai_msg = AIMessage(
    content = [],
    tool_calls = [{
        "name": "get_weather",
        "args": {"location": "New York"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "The weather in New York is sunny and 75°F."
tool_msg = ToolMessage(
    content = weather_result,
    tool_call_id = "call_123"   # Must match the call ID
)

# Continue the conversation with the tool result
messages = [
    HumanMessage("What's the weather in New York?"),
    ai_msg,   # Model's tool call
    tool_msg, # Result of the tool call
]
response = model.invoke(messages)

In [13]:
print(response.content)

<think>
Okay, the user asked for the weather in New York. I called the get_weather function with the location set to New York. The response came back saying it's sunny and 75°F. Now I need to present this information clearly.

First, I should confirm the location to make sure there's no confusion. Then, state the current conditions: sunny with the temperature. Maybe add a friendly note about it being a good day to be outdoors. Keep it concise and straightforward. Let me check if there's any additional details needed, but since the user only asked for the weather, sticking to the provided info should be fine. Alright, time to put it all together in a natural, conversational way.
</think>

The current weather in New York is **sunny** with temperatures at **75°F**. It looks like a great day to enjoy outdoor activities! 🌞
